# Hybrid Retrieval

BM25 and dense retrieval fail in different places. BM25 needs the question to repeat words from the passage, so it finds "research and development" on the page that says exactly that and misses the page that calls it "technical infrastructure investment". Dense retrieval compares meaning and gets that second page, but it can wander off to a passage that is about the right topic and never states the number.

Hybrid retrieval runs both and merges the two ranked lists with reciprocal rank fusion. Every passage gets `1 / (60 + rank)` from each list it appears in and the two are added. A passage that sits near the top of both lists ends up above one that is first in one list and missing from the other. Only ranks are compared, never scores, so a BM25 score and a cosine similarity never have to be put on the same scale.

## Load in documents and chunk them

The chunks are the same ones the dense, HyDE and graph notebooks use, so the pages each retriever returns can be lined up across notebooks.

In [1]:
from rag.documents import load_documents
from rag.chunk import chunk_documents

FILE_PATH = "/home/nick/github-projects/Sec-Rag/data/google_10K.pdf"

documents = load_documents(FILE_PATH)

chunks = chunk_documents(
    documents=documents,
    chunk_size=400,
    chunk_overlap=40
)

print(f"Length of Documents: {len(documents)}")
print(f"Length of Chunks: {len(chunks)}")

/home/nick/github-projects/Sec-Rag/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Length of Documents: 107
Length of Chunks: 1037


## Build each half on its own

`HybridRetriever` will build its own halves if it is given none, but it also accepts ones that are already indexed. Building them here first means each half can be queried by itself before they are fused, and nothing gets embedded twice.

In [2]:
from rag.bm25 import BM25Retriever
from rag.dense import DenseRetriever

bm25 = BM25Retriever()
bm25.add_documents(chunks)

dense = DenseRetriever()
dense.add_documents(chunks)

print(bm25)
print(dense)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5806.24it/s]


BM25Retriever(documents=1037)
DenseRetriever(model_name='sentence-transformers/all-MiniLM-L6-v2', documents=1037)


## See where the halves disagree

One query, top five from each half. The overlap is what both methods are sure about; the rest is where they pull apart.

In [3]:
def pages(docs):
    return [d.metadata["page"] for d in docs]


query = "how much did Google spend on research and development in 2025"

bm25_docs = bm25.retrieve(query, top_k=4)
dense_docs = dense.retrieve(query, top_k=4)

print(f"bm25 pages:  {pages(bm25_docs)}")
print(f"dense pages: {pages(dense_docs)}")

print("\n--- bm25 ---")
for doc in bm25_docs:
    print(f"\n[page {doc.metadata['page']}]\n{doc.page_content[:200]}")

print("\n--- dense ---")
for doc in dense_docs:
    print(f"\n[page {doc.metadata['page']}]\n{doc.page_content[:200]}")

bm25 pages:  [32, 41, 3, 42]
dense pages: [41, 40, 39, 41]

--- bm25 ---

[page 32]
Alphabet is a collection of businesses — the largest of which is Google. We report Google in two segments,
Google Services and Google Cloud, and all non-Google businesses collectively as Other Bets. S

[page 41]
costs, largely for YouTube, depreciation expense, and other technical infrastructure operations costs.
Research and Development
The following table presents research and development expenses (in milli

[page 3]
Sundar Pichai.
Alphabet is a collection of businesses — the largest of which is Google. We report Google in two segments,
Google Services and Google Cloud, and all non-Google businesses collectively a

[page 42]
Other Bets (4,444) (7,515)
Alphabet-level activities (10,541) (16,760)
Total income from operations $ 112,390  $ 129,039 
Alphabet-level activities primarily reflect expenses related to our shared AI 

--- dense ---

[page 41]
2025, primarily due to a revenue mix shift from Google

## Fuse them

Both halves go into `HybridRetriever`. Nothing is re-indexed; it only calls `retrieve` on each. `candidates=40` is how far down each list it looks before fusing, and `rrf_k=60` is the offset in `1 / (60 + rank)` that stops the first rank from swamping everything below it. A page chunked into several passages only comes back once, as its best-ranked chunk, so the top five are five different pages.

In [4]:
from rag.hybrid import HybridRetriever

hybrid = HybridRetriever(bm25=bm25, dense=dense)

hybrid_docs = hybrid.retrieve(query, top_k=4)

print(f"bm25 pages:   {pages(bm25_docs)}")
print(f"dense pages:  {pages(dense_docs)}")
print(f"hybrid pages: {pages(hybrid_docs)}")

for doc in hybrid_docs:
    print(f"\n[page {doc.metadata['page']}]\n{doc.page_content[:200]}")

bm25 pages:   [32, 41, 3, 42]
dense pages:  [41, 40, 39, 41]
hybrid pages: [41, 42, 32, 39]

[page 41]
costs, largely for YouTube, depreciation expense, and other technical infrastructure operations costs.
Research and Development
The following table presents research and development expenses (in milli

[page 42]
Other Bets (4,444) (7,515)
Alphabet-level activities (10,541) (16,760)
Total income from operations $ 112,390  $ 129,039 
Alphabet-level activities primarily reflect expenses related to our shared AI 

[page 32]
Alphabet is a collection of businesses — the largest of which is Google. We report Google in two segments,
Google Services and Google Cloud, and all non-Google businesses collectively as Other Bets. S

[page 39]
Total revenues $ 350,018  $ 402,836 
Google Services
Google Advertising
Google Search & other
Google Search & other revenues increased $26.4 billion from 2024 to 2025. The overall growth was driven by


## Weight one half

`bm25_weight` and `dense_weight` scale what each half adds. Doubling the BM25 weight leans the fused list toward exact wording; doubling the dense weight leans it toward meaning. The halves are the same objects, so this is cheap to try.

In [5]:
lean_bm25 = HybridRetriever(bm25=bm25, dense=dense, bm25_weight=2.0)
lean_dense = HybridRetriever(bm25=bm25, dense=dense, dense_weight=2.0)

print(f"equal:      {pages(hybrid.retrieve(query, top_k=4))}")
print(f"bm25 x2:    {pages(lean_bm25.retrieve(query, top_k=4))}")
print(f"dense x2:   {pages(lean_dense.retrieve(query, top_k=4))}")

equal:      [41, 42, 32, 39]
bm25 x2:    [41, 32, 42, 10]
dense x2:   [41, 39, 42, 32]


## Compare bm25, dense and hybrid, then answer

For each question the pages from each half are printed next to the pages the hybrid returns, followed by the answer generated over the hybrid context. `RAGPipeline` only needs something with a `retrieve` method. The generator is `gpt-4o-mini` behind `LangChainGenerator`, with `OPENAI_API_KEY` read from `.env` by `load_dotenv()`.

In [6]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from rag.llm import DEFAULT_CHAT_MODEL, LangChainGenerator
from rag.pipeline import RAGPipeline

load_dotenv()

generator = LangChainGenerator(ChatOpenAI(model=DEFAULT_CHAT_MODEL, max_tokens=512))
pipeline = RAGPipeline(hybrid, generator, top_k=4)

for i, q in enumerate([
    "how much did Google spend on research and development in 2025?",
    "how much did Google spend on research and development in 2024?",
    "what were Google's main operating costs?",
    "what are some pending acquisitions the company faces?",
]):
    answer = pipeline.answer(q)
    print(f"\n━━━ {i + 1} Q: {q}")
    print(f"bm25 pages:   {pages(bm25.retrieve(q, top_k=4))}")
    print(f"dense pages:  {pages(dense.retrieve(q, top_k=4))}")
    print(f"hybrid pages: {pages(answer.documents)}")
    print(f"> {answer.text}")


━━━ 1 Q: how much did Google spend on research and development in 2025?
bm25 pages:   [32, 41, 3, 42]
dense pages:  [40, 41, 41, 39]
hybrid pages: [41, 42, 39, 32]
> Google spent $61,087 million on research and development in 2025. [Source: /home/nick/github-projects/Sec-Rag/data/google_10K.pdf p41]

━━━ 2 Q: how much did Google spend on research and development in 2024?
bm25 pages:   [41, 32, 3, 42]
dense pages:  [40, 41, 41, 39]
hybrid pages: [41, 42, 39, 32]
> Google spent $49,326 million on research and development in 2024. 

(Source: /home/nick/github-projects/Sec-Rag/data/google_10K.pdf p41)

━━━ 3 Q: what were Google's main operating costs?
bm25 pages:   [36, 83, 4, 42]
dense pages:  [94, 38, 42, 94]
hybrid pages: [42, 38, 94, 95]
> Google's main operating costs included traffic acquisition costs (TAC), content acquisition costs, and depreciation expense. Additionally, overall costs and expenses related to Google Cloud, such as technical infrastructure and office facilities usa